In [24]:
# ===== SEL 1: SETUP, HEADER, SESI HTTP, DAN FUNGSI REQUEST AMAN =====
from pathlib import Path
from datetime import datetime, timezone, date, timedelta
import calendar, hashlib, json, re, time, zipfile
import requests
import pandas as pd
from bs4 import BeautifulSoup
from urllib.robotparser import RobotFileParser
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

# GANTI dengan email kampus Anda yang sebenarnya
EMAIL_KAMPUS = "riska@mahasiswa.instiki.ac.id"
assert "mahasiswa@instiki.ac.id" not in EMAIL_KAMPUS, "mahasiswa@instiki.ac.id"

# User-Agent mengidentifikasi aplikasi dan kontak (syarat soal)
HEADERS = {"User-Agent": f"UTS-DAM-NTB/1.0 (kontak: {EMAIL_KAMPUS})"}
TIMEOUT = 30        # detik, wajib pada setiap request
JEDA_DETIK = 1.5    # jeda antar-request

# Struktur folder proyek
for folder in ["data/raw", "data/interim", "data/processed", "metadata", "docs"]:
    Path(folder).mkdir(parents=True, exist_ok=True)

# Sesi HTTP dengan retry otomatis untuk 429 dan 5xx
session = requests.Session()
retry_config = Retry(total=3, connect=3, read=3, backoff_factor=1,
                     status_forcelist=[429, 500, 502, 503, 504],
                     allowed_methods=["GET"], respect_retry_after_header=True)
adapter = HTTPAdapter(max_retries=retry_config)
session.mount("https://", adapter)
session.mount("http://", adapter)

# Catatan waktu akses asli tiap berkas mentah (dipakai di provenance)
log_akses = {}

def get_aman(url, **kwargs):
    """GET dengan timeout + User-Agent + jeda. Mengembalikan (response, waktu_akses_utc)."""
    kwargs.setdefault("headers", HEADERS)
    kwargs.setdefault("timeout", TIMEOUT)
    resp = session.get(url, **kwargs)
    waktu = datetime.now(timezone.utc).isoformat()
    time.sleep(JEDA_DETIK)
    return resp, waktu

def hitung_md5(path):
    """Hash MD5 berkas untuk provenance."""
    d = hashlib.md5()
    with open(path, "rb") as f:
        for blok in iter(lambda: f.read(8192), b""):
            d.update(blok)
    return d.hexdigest()

bulan_map = {"Januari":1,"Februari":2,"Maret":3,"April":4,"Mei":5,"Juni":6,
             "Juli":7,"Agustus":8,"September":9,"Oktober":10,"November":11,"Desember":12}
print("Setup siap.")

Setup siap.


In [25]:
# ===== SEL 2: UNGGAH CSV MENTAH KE data/raw (ISI TIDAK DIUBAH) =====
from google.colab import files
uploaded = files.upload()

nama_asli = next(iter(uploaded))
# Hilangkan akhiran " (1)" dari nama file; hanya nama yang berubah, isi byte tetap
nama_bersih = re.sub(r"\s\(\d+\)(?=\.csv$)", "", nama_asli)
raw_dataset_path = Path("data/raw") / nama_bersih
raw_dataset_path.write_bytes(uploaded[nama_asli])
log_akses[nama_bersih] = datetime.now(timezone.utc).isoformat()

URL_DATASET = "https://data.ntbprov.go.id/dataset/9e270f88-a604-4640-a4f0-24c996122b92/show"
print("Tersimpan:", raw_dataset_path, "|", raw_dataset_path.stat().st_size, "byte")

Saving data_20261119132258-jumlah-kunjungan-wisatawan-berdasarkan-jenis-wisatawan.csv to data_20261119132258-jumlah-kunjungan-wisatawan-berdasarkan-jenis-wisatawan (2).csv
Tersimpan: data/raw/data_20261119132258-jumlah-kunjungan-wisatawan-berdasarkan-jenis-wisatawan.csv | 24386 byte


In [33]:
# ===== SEL 3 (SOAL 1): BACA CSV, IDENTIFIKASI, ESTIMASI STORAGE =====
# Baca CSV dengan parameter eksplisit; Jumlah dibaca sebagai teks dulu agar pemisah ribuan aman
df = pd.read_csv(
    raw_dataset_path, sep=",", encoding="utf-8",
    dtype={"#":"Int64","Kode provinsi":"Int64","Nama Provinsi":"string",
           "Kategori Wisatawan":"string","Jumlah (Orang)":"string",
           "Tahun":"Int64","Bulan":"string"},
    na_values=["", "NA", "N/A", "-"]
)
df.columns = df.columns.str.strip()
for k in ["Nama Provinsi", "Kategori Wisatawan", "Bulan"]:
    df[k] = df[k].str.strip()
# Hapus semua karakter non-angka (koma/titik ribuan) lalu ubah ke numerik
df["Jumlah (Orang)"] = pd.to_numeric(
    df["Jumlah (Orang)"].str.replace(r"[^\d-]", "", regex=True), errors="coerce")

print("Kategori SEBELUM dibersihkan:\n", df["Kategori Wisatawan"].value_counts())

# Salinan untuk pembersihan; df tetap menjadi data apa adanya dari file
df_bersih = df.copy()
df_bersih["Kategori Wisatawan"] = df_bersih["Kategori Wisatawan"].replace(
    {"Wisatawan Mancaegara": "Wisatawan Mancanegara"})   # perbaiki salah ketik
print("\nKategori SESUDAH dibersihkan:\n", df_bersih["Kategori Wisatawan"].value_counts())
assert df_bersih["Kategori Wisatawan"].nunique() == 2
df_bersih.to_csv("data/interim/kunjungan_wisatawan_ntb_bersih.csv", index=False)

display(df.head()); print(df.shape); print(df.dtypes); print(df.isna().sum())

# Identifikasi dataset + estimasi storage
ukuran_byte = raw_dataset_path.stat().st_size
ukuran_mb = ukuran_byte / 1024**2
# Estimasi kebutuhan storage selama 1 tahun
# Sesuai soal: ukuran berkas x 12 bulan + cadangan 30%
estimasi_12_bulan_mb = ukuran_mb * 12
cadangan_30_persen_mb = estimasi_12_bulan_mb * 0.30
estimasi_mb = estimasi_12_bulan_mb + cadangan_30_persen_mb

print("===== ESTIMASI STORAGE =====")
print(f"Ukuran file asli       : {ukuran_mb:.4f} MB")
print(f"Estimasi 12 bulan      : {estimasi_12_bulan_mb:.4f} MB")
print(f"Cadangan 30%           : {cadangan_30_persen_mb:.4f} MB")
print(f"Total estimasi 1 tahun : {estimasi_mb:.4f} MB")

identifikasi = pd.DataFrame([{
    "jenis_data": "Terstruktur (tabular CSV)",
    "jumlah_baris": len(df), "jumlah_kolom": df.shape[1],
    "tipe_data": ", ".join(f"{c}: {t}" for c, t in df.dtypes.items()),
    "satuan_jumlah": "Orang (kunjungan wisatawan)",
    "frekuensi": "Bulanan, 2 kategori wisatawan per bulan",
    "cakupan_wilayah": ", ".join(df["Nama Provinsi"].unique()),
    "cakupan_waktu": f"{df['Tahun'].min()}-{df['Tahun'].max()}",
    "ukuran_byte": ukuran_byte, "ukuran_MB": round(ukuran_mb, 4),
    "estimasi_storage_1_tahun_MB": round(estimasi_mb, 4),
   "asumsi_storage": "Ukuran file dianggap mewakili kebutuhan data selama 1 bulan, kemudian dikalikan 12 bulan dan ditambahkan cadangan storage sebesar 30%."
}])
display(identifikasi.T)
identifikasi.to_csv("metadata/identifikasi_dataset.csv", index=False)

Kategori SEBELUM dibersihkan:
 Kategori Wisatawan
Wisatawan Nusantara      150
Wisatawan Mancanegara    149
Wisatawan Mancaegara       1
Name: count, dtype: Int64

Kategori SESUDAH dibersihkan:
 Kategori Wisatawan
Wisatawan Nusantara      150
Wisatawan Mancanegara    150
Name: count, dtype: Int64


,#,Kode provinsi,Nama Provinsi,Kategori Wisatawan,Jumlah (Orang),Tahun,Bulan
0,1,52,Nusa Tenggara Barat,Wisatawan Nusantara,132472,2026,Juni
1,2,52,Nusa Tenggara Barat,Wisatawan Mancanegara,88315,2026,Juni
2,3,52,Nusa Tenggara Barat,Wisatawan Nusantara,122674,2026,Mei
3,4,52,Nusa Tenggara Barat,Wisatawan Mancanegara,81783,2026,Mei
4,5,52,Nusa Tenggara Barat,Wisatawan Nusantara,113304,2026,April


(300, 7)
#                              Int64
Kode provinsi                  Int64
Nama Provinsi         string[python]
Kategori Wisatawan    string[python]
Jumlah (Orang)                 Int64
Tahun                          Int64
Bulan                 string[python]
dtype: object
#                     0
Kode provinsi         0
Nama Provinsi         0
Kategori Wisatawan    0
Jumlah (Orang)        0
Tahun                 0
Bulan                 0
dtype: int64
===== ESTIMASI STORAGE =====
Ukuran file asli       : 0.0233 MB
Estimasi 12 bulan      : 0.2791 MB
Cadangan 30%           : 0.0837 MB
Total estimasi 1 tahun : 0.3628 MB


,0
jenis_data,Terstruktur (tabular CSV)
jumlah_baris,300
jumlah_kolom,7
tipe_data,"#: Int64, Kode provinsi: Int64, Nama Provinsi:..."
satuan_jumlah,Orang (kunjungan wisatawan)
frekuensi,"Bulanan, 2 kategori wisatawan per bulan"
cakupan_wilayah,Nusa Tenggara Barat
cakupan_waktu,2014-2026
ukuran_byte,24386
ukuran_MB,0.0233


In [27]:

# ===== SEL 4 (SOAL 2): LISENSI, PII, DATA ETHICS CHECKLIST 9 BUTIR =====

from datetime import datetime
from pathlib import Path
import pandas as pd

# Identitas dataset
NAMA_DATASET = "Jumlah Kunjungan Wisatawan berdasarkan Jenis Wisatawan"
TGL = datetime.now().strftime("%Y-%m-%d")

URL_DATASET = (
    "https://data.ntbprov.go.id/dataset/"
    "9e270f88-a604-4640-a4f0-24c996122b92/show"
)

# Pastikan folder metadata tersedia
Path("metadata").mkdir(parents=True, exist_ok=True)

# ==================================================
# 1. BUKTI LISENSI DAN KETENTUAN PORTAL
# ==================================================

BUKTI_LISENSI = (
    "Halaman dataset Satu Data NTB menampilkan keterangan "
    "'License Not Specified'. "
    f"URL: {URL_DATASET}. "
    f"Tanggal pemeriksaan: {TGL}. "
    "Lisensi penggunaan ulang belum dapat dipastikan."
)

BUKTI_TOS = (
    "Ketentuan penggunaan atau Terms of Service portal "
    "belum berhasil diverifikasi. "
    "Tidak ditemukan bukti ketentuan yang dapat digunakan "
    "untuk menyatakan izin penggunaan ulang. "
    f"Portal sumber: {URL_DATASET}. "
    f"Tanggal pemeriksaan: {TGL}."
)

# ==================================================
# 2. KLASIFIKASI PII DAN QUASI-IDENTIFIER
# ==================================================

klasifikasi_pii = pd.DataFrame([
    ["#", "Bukan PII",
     "Nomor urut baris"],

    ["Kode provinsi", "Quasi-identifier (geografis)",
     "Level provinsi, 1 nilai tunggal (52), risiko re-identifikasi sangat rendah"],

    ["Nama Provinsi", "Quasi-identifier (geografis)",
     "Sama seperti di atas"],

    ["Kategori Wisatawan", "Atribut kategorikal agregat",
     "Hanya 2 kategori, bukan atribut individu"],

    ["Jumlah (Orang)", "Nilai agregat",
     "Puluhan ribu orang per baris, tidak ada individu"],

    ["Tahun", "Quasi-identifier (temporal)",
     "Granularitas bulanan, agregat"],

    ["Bulan", "Quasi-identifier (temporal)",
     "Granularitas bulanan, agregat"]
], columns=["Kolom", "Klasifikasi", "Alasan"])

display(klasifikasi_pii)

klasifikasi_pii.to_csv(
    "metadata/klasifikasi_kolom_pii.csv",
    index=True,
    encoding="utf-8-sig"
)

# ==================================================
# 3. DATA ETHICS CHECKLIST 9 BUTIR
# ==================================================

rows = [
    [
        1,
        "Sumber dan URL dataset",
        "TERPENUHI",
        f"{URL_DATASET} | diakses {TGL}",
        f"Dataset '{NAMA_DATASET}' berasal dari Satu Data Provinsi NTB."
    ],
    [
        2,
        "Lisensi dataset",
        "DIVERIFIKASI",
        BUKTI_LISENSI,
        "Lisensi tidak dinyatakan pada halaman dataset. "
        "Penggunaan untuk tugas akademik disertai atribusi sumber, "
        "tetapi izin penggunaan ulang belum dapat dipastikan."
    ],
    [
        3,
        "Ketentuan portal / ToS",
        "DIVERIFIKASI",
        BUKTI_TOS,
        "Perlu konfirmasi ketentuan penggunaan portal. "
        "Tidak menganggap data yang dapat diunduh otomatis bebas digunakan."
    ],
    [
        4,
        "Identifikasi PII dan quasi-identifier",
        "TERPENUHI",
        "Lihat metadata/klasifikasi_kolom_pii.csv. "
        "Sebanyak 7 kolom telah diperiksa.",
        "Tidak ditemukan PII individu. "
        "Quasi-identifier berupa informasi geografis dan temporal "
        "pada data agregat."
    ],
    [
        5,
        "Penanganan PII",
        "TIDAK DIPERLUKAN",
        "Hasil pemeriksaan butir 4 menunjukkan tidak ada PII individu.",
        "Jika ditemukan PII pada sumber lain, data harus dihapus "
        "atau dianonimkan sebelum dianalisis."
    ],
    [
        6,
        "Data minimization",
        "TERPENUHI",
        "Data cuaca menggunakan 3 variabel: suhu, curah hujan, "
        "dan kecepatan angin.",
        "Variabel yang diambil disesuaikan dengan kebutuhan analisis "
        "dan tidak mengambil atribut yang tidak diperlukan."
    ],
    [
        7,
        "Potensi bias data",
        "TERPENUHI DENGAN CATATAN",
        "Dataset kunjungan merupakan data agregat tingkat provinsi. "
        "Data cuaca diambil dari satu titik koordinat di Mataram.",
        "Cuaca Mataram belum tentu mewakili seluruh wilayah NTB "
        "seperti Sumbawa dan Bima. "
        "Metode penghitungan kunjungan oleh pengelola data "
        "juga tidak dijelaskan secara rinci."
    ],
    [
        8,
        "Tujuan penggunaan",
        "TERPENUHI",
        "Tugas UTS Data Acquisition & Management (INSTIKI).",
        "Data digunakan untuk kepentingan akademik, "
        "bukan untuk mengidentifikasi individu atau tujuan komersial."
    ],
    [
        9,
        "Retensi dan penyimpanan",
        "TERPENUHI",
        "Data mentah disimpan di data/raw, "
        "hasil pengolahan di data/processed, "
        "dan dokumentasi di metadata/.",
        "Salinan data dikelola untuk kebutuhan tugas "
        "dan dihapus setelah tidak lagi diperlukan sesuai ketentuan."
    ]
]

ethics = pd.DataFrame(
    rows,
    columns=[
        "No",
        "Butir Checklist",
        "Status",
        "Bukti",
        "Catatan / Tindakan"
    ]
)

# ==================================================
# 4. VALIDASI BUKTI DAN STATUS
# ==================================================

# Periksa bukti yang kosong
bukti_kosong = ethics[
    ethics["Bukti"].isna() |
    ethics["Bukti"].astype(str).str.strip().eq("")
]

# Tampilkan hasil checklist
display(ethics)

# Simpan hasil ke CSV dan Excel
ethics.to_csv(
    "metadata/data_ethics_checklist.csv",
    index=False,
    encoding="utf-8-sig"
)

ethics.to_excel(
    "metadata/data_ethics_checklist.xlsx",
    index=False
)

# ==================================================
# 5. RINGKASAN HASIL PEMERIKSAAN
# ==================================================

print("========== RINGKASAN DATA ETHICS ==========")
print("Jumlah butir checklist :", len(ethics))
print("Butir terpenuhi        :",
      (ethics["Status"] == "TERPENUHI").sum())
print("Terpenuhi dengan catatan:",
      (ethics["Status"] == "TERPENUHI DENGAN CATATAN").sum())
print("Tidak diperlukan       :",
      (ethics["Status"] == "TIDAK DIPERLUKAN").sum())
print("Belum diverifikasi     :",
      (ethics["Status"] == "BELUM DIVERIFIKASI").sum())

if not bukti_kosong.empty:
    print("\nPERINGATAN: Masih terdapat bukti yang kosong.")
else:
    print("\nSemua kolom bukti telah diisi.")

print("\nFile berhasil disimpan:")
print("- metadata/klasifikasi_kolom_pii.csv")
print("- metadata/data_ethics_checklist.csv")
print("- metadata/data_ethics_checklist.xlsx")


,Kolom,Klasifikasi,Alasan
0,#,Bukan PII,Nomor urut baris
1,Kode provinsi,Quasi-identifier (geografis),"Level provinsi, 1 nilai tunggal (52), risiko r..."
2,Nama Provinsi,Quasi-identifier (geografis),Sama seperti di atas
3,Kategori Wisatawan,Atribut kategorikal agregat,"Hanya 2 kategori, bukan atribut individu"
4,Jumlah (Orang),Nilai agregat,"Puluhan ribu orang per baris, tidak ada individu"
5,Tahun,Quasi-identifier (temporal),"Granularitas bulanan, agregat"
6,Bulan,Quasi-identifier (temporal),"Granularitas bulanan, agregat"


,No,Butir Checklist,Status,Bukti,Catatan / Tindakan
0,1,Sumber dan URL dataset,TERPENUHI,https://data.ntbprov.go.id/dataset/9e270f88-a6...,Dataset 'Jumlah Kunjungan Wisatawan berdasarka...
1,2,Lisensi dataset,DIVERIFIKASI,Halaman dataset Satu Data NTB menampilkan kete...,Lisensi tidak dinyatakan pada halaman dataset....
2,3,Ketentuan portal / ToS,DIVERIFIKASI,Ketentuan penggunaan atau Terms of Service por...,Perlu konfirmasi ketentuan penggunaan portal. ...
3,4,Identifikasi PII dan quasi-identifier,TERPENUHI,Lihat metadata/klasifikasi_kolom_pii.csv. Seba...,Tidak ditemukan PII individu. Quasi-identifier...
4,5,Penanganan PII,TIDAK DIPERLUKAN,Hasil pemeriksaan butir 4 menunjukkan tidak ad...,"Jika ditemukan PII pada sumber lain, data haru..."
5,6,Data minimization,TERPENUHI,"Data cuaca menggunakan 3 variabel: suhu, curah...",Variabel yang diambil disesuaikan dengan kebut...
6,7,Potensi bias data,TERPENUHI DENGAN CATATAN,Dataset kunjungan merupakan data agregat tingk...,Cuaca Mataram belum tentu mewakili seluruh wil...
7,8,Tujuan penggunaan,TERPENUHI,Tugas UTS Data Acquisition & Management (INSTI...,"Data digunakan untuk kepentingan akademik, buk..."
8,9,Retensi dan penyimpanan,TERPENUHI,"Data mentah disimpan di data/raw, hasil pengol...",Salinan data dikelola untuk kebutuhan tugas da...


========== RINGKASAN DATA ETHICS ==========
Jumlah butir checklist : 9
Butir terpenuhi        : 5
Terpenuhi dengan catatan: 1
Tidak diperlukan       : 1
Belum diverifikasi     : 0

Semua kolom bukti telah diisi.

File berhasil disimpan:
- metadata/klasifikasi_kolom_pii.csv
- metadata/data_ethics_checklist.csv
- metadata/data_ethics_checklist.xlsx


In [28]:
# ===== SEL 5 (SOAL 3): API CUACA HISTORIS OPEN-METEO (tanpa API key) =====
# Open-Meteo: CC BY 4.0, gratis nonkomersial. Wajib atribusi: "Weather data by Open-Meteo.com"
API_URL = "https://archive-api.open-meteo.com/v1/archive"

# Periode API mengikuti rentang tahun-bulan dataset (variabel lokal, df tidak diubah)
bulan_num = df_bersih["Bulan"].map(bulan_map)
assert bulan_num.notna().all(), "Ada nama bulan yang tidak cocok dengan bulan_map."
periode = df_bersih["Tahun"].astype(int) * 100 + bulan_num.astype(int)
p_min, p_max = int(periode.min()), int(periode.max())
tanggal_awal = date(p_min // 100, p_min % 100, 1)
tanggal_akhir = date(p_max // 100, p_max % 100, calendar.monthrange(p_max // 100, p_max % 100)[1])
tanggal_akhir = min(tanggal_akhir, date.today() - timedelta(days=5))  # arsip punya jeda beberapa hari

params = {
    "latitude": -8.5833, "longitude": 116.1167,          # Kota Mataram (titik wakil NTB)
    "start_date": tanggal_awal.isoformat(), "end_date": tanggal_akhir.isoformat(),
    "daily": "temperature_2m_mean,precipitation_sum,wind_speed_10m_max",
    "timezone": "Asia/Makassar",
}

# Penanganan status code + retry (retry otomatis lewat session; sisanya ditangani di sini)
try:
    resp, waktu_api = get_aman(API_URL, params=params)
    print("HTTP status:", resp.status_code)
    resp.raise_for_status()
except requests.exceptions.HTTPError:
    kode = resp.status_code
    if kode == 429:
        print("Rate limit terlampaui. Tunggu lalu jalankan ulang.")
    elif 400 <= kode < 500:
        print("Parameter salah:", resp.text[:300])
    else:
        print("Server bermasalah setelah 3x retry:", kode)
    raise
except requests.exceptions.RequestException as e:   # timeout, koneksi, RetryError
    print("Gagal koneksi/timeout:", e)
    raise

# Simpan respons mentah apa adanya (bytes) dan catat waktu akses
raw_api_path = Path("data/raw/open_meteo_ntb_historis.json")
raw_api_path.write_bytes(resp.content)
log_akses[raw_api_path.name] = waktu_api

# Normalisasi JSON -> tabel
payload = resp.json()
print("Satuan dari API:", payload.get("daily_units"))
daily = payload["daily"]
cuaca = pd.DataFrame({
    "tanggal": pd.to_datetime(daily["time"]),
    "suhu_rata_rata_c": daily["temperature_2m_mean"],
    "curah_hujan_mm": daily["precipitation_sum"],
    "kecepatan_angin_maks_kmh": daily["wind_speed_10m_max"],
})
cuaca["Tahun"] = cuaca["tanggal"].dt.year
cuaca["Bulan"] = cuaca["tanggal"].dt.month
assert len(cuaca) >= 30, "Data API kurang dari 30 baris."
print("Nilai kosong:\n", cuaca.isna().sum())
cuaca.to_csv("data/interim/cuaca_harian_ntb_historis.csv", index=False)
display(cuaca.head()); print("Baris:", len(cuaca), "| Kolom:", cuaca.shape[1])
print("Atribusi: Weather data by Open-Meteo.com (CC BY 4.0) https://open-meteo.com/en/license")

HTTP status: 200
Satuan dari API: {'time': 'iso8601', 'temperature_2m_mean': '°C', 'precipitation_sum': 'mm', 'wind_speed_10m_max': 'km/h'}
Nilai kosong:
 tanggal                     0
suhu_rata_rata_c            0
curah_hujan_mm              0
kecepatan_angin_maks_kmh    0
Tahun                       0
Bulan                       0
dtype: int64


,tanggal,suhu_rata_rata_c,curah_hujan_mm,kecepatan_angin_maks_kmh,Tahun,Bulan
0,2014-01-01,24.8,63.3,22.1,2014,1
1,2014-01-02,25.1,30.3,18.3,2014,1
2,2014-01-03,25.6,30.7,8.6,2014,1
3,2014-01-04,25.3,14.0,12.6,2014,1
4,2014-01-05,25.4,22.9,9.3,2014,1


Baris: 4564 | Kolom: 6
Atribusi: Weather data by Open-Meteo.com (CC BY 4.0) https://open-meteo.com/en/license


In [29]:
# ===== SEL 6 (SOAL 4): CEK ROBOTS.TXT + SCRAPING TABEL WIKIPEDIA =====
from io import StringIO

ROBOTS_URL = "https://id.wikipedia.org/robots.txt"
# Halaman statis kandidat; dipakai yang pertama berstatus 200 dan berisi tabel
KANDIDAT_URL = [
    "https://id.wikipedia.org/wiki/Daftar_kabupaten_dan_kota_di_Nusa_Tenggara_Barat",
    "https://id.wikipedia.org/wiki/Nusa_Tenggara_Barat",
]

# 1) Ambil dan simpan robots.txt mentah sebagai bukti
r_robots, waktu_robots = get_aman(ROBOTS_URL)
r_robots.raise_for_status()
Path("data/raw/wikipedia_robots.txt").write_bytes(r_robots.content)
log_akses["wikipedia_robots.txt"] = waktu_robots

# 2) Parser robots.txt untuk User-Agent kita dan wildcard
rp = RobotFileParser()
rp.parse(r_robots.text.splitlines())
ua = HEADERS["User-Agent"]

# 3) Cari halaman yang diizinkan, ada (200), dan berisi tabel
SCRAPE_URL, r_page, waktu_page, tabel_list = None, None, None, []
for url in KANDIDAT_URL:
    izin = rp.can_fetch(ua, url) and rp.can_fetch("*", url)
    print(f"{url} -> diizinkan robots.txt: {izin}")
    if not izin:
        continue
    resp, waktu = get_aman(url)
    print("   HTTP status:", resp.status_code)
    if resp.status_code != 200:
        continue
    try:
        tabel_list = pd.read_html(StringIO(resp.text))   # parsing semua <table>
    except ValueError:
        tabel_list = []
    if tabel_list:
        SCRAPE_URL, r_page, waktu_page = url, resp, waktu
        break
if SCRAPE_URL is None:
    raise SystemExit("Tidak ada halaman kandidat yang valid. Ganti KANDIDAT_URL.")

# 4) Simpan HTML mentah persis seperti diterima (tidak diubah)
Path("data/raw/wikipedia_ntb.html").write_bytes(r_page.content)
log_akses["wikipedia_ntb.html"] = waktu_page

# 5) Bukti kepatuhan robots.txt: hasil DIHITUNG dari parser, bukan ditulis manual
izin_ua, izin_all = rp.can_fetch(ua, SCRAPE_URL), rp.can_fetch("*", SCRAPE_URL)
bukti_robots = {
    "situs": "id.wikipedia.org", "url_robots": ROBOTS_URL, "url_halaman": SCRAPE_URL,
    "http_status_robots": r_robots.status_code, "waktu_pemeriksaan_utc": waktu_robots,
    "md5_robots_txt": hitung_md5("data/raw/wikipedia_robots.txt"),
    "user_agent": ua, "can_fetch_user_agent": izin_ua, "can_fetch_wildcard": izin_all,
    "crawl_delay": rp.crawl_delay(ua), "jeda_yang_dipakai_detik": JEDA_DETIK,
    "hasil": "DIIZINKAN" if (izin_ua and izin_all) else "DILARANG",
}
assert bukti_robots["hasil"] == "DIIZINKAN", "robots.txt tidak mengizinkan halaman ini."
Path("metadata/bukti_robots_wikipedia.json").write_text(
    json.dumps(bukti_robots, indent=4, ensure_ascii=False), encoding="utf-8")
print(json.dumps(bukti_robots, indent=4, ensure_ascii=False))

# 6) ID revisi artikel untuk atribusi dan provenance
m = re.search(r'"wgRevisionId":(\d+)', r_page.text)
revisi_id = m.group(1) if m else "tidak ditemukan"
print("URL:", SCRAPE_URL, "| ID revisi:", revisi_id)

# 7) Pilih tabel: tampilkan semua, ambil yang barisnya terbanyak
for i, t in enumerate(tabel_list):
    print(f"Tabel {i}: {t.shape[0]} baris x {t.shape[1]} kolom")
IDX_TABEL = max(range(len(tabel_list)), key=lambda i: tabel_list[i].shape[0])
# Jika kurang relevan, isi manual lalu jalankan ulang sel ini, mis. IDX_TABEL = 1

# 8) Rapikan tabel: gabung header bertingkat, buang penanda catatan kaki seperti [1]
scrape_df = tabel_list[IDX_TABEL].copy()
if isinstance(scrape_df.columns, pd.MultiIndex):
    scrape_df.columns = [" ".join(dict.fromkeys(map(str, c))).strip() for c in scrape_df.columns]
scrape_df.columns = [re.sub(r"\[.*?\]", "", str(c)).strip() for c in scrape_df.columns]
for kol in scrape_df.select_dtypes(include="object").columns:
    scrape_df[kol] = scrape_df[kol].astype(str).str.replace(r"\[.*?\]", "", regex=True).str.strip()
scrape_df = scrape_df.drop_duplicates().reset_index(drop=True)
scrape_df["sumber_halaman"] = SCRAPE_URL
scrape_df["id_revisi_wikipedia"] = revisi_id

# 9) Simpan hasil scraping
scrape_df.to_csv("data/interim/tabel_wikipedia_ntb_scraping.csv", index=False)
print("Baris:", len(scrape_df), "| Kolom:", scrape_df.shape[1])
display(scrape_df.head(15))
print("Atribusi: Wikipedia Bahasa Indonesia, kontributor Wikipedia, " + SCRAPE_URL)

https://id.wikipedia.org/wiki/Daftar_kabupaten_dan_kota_di_Nusa_Tenggara_Barat -> diizinkan robots.txt: True
   HTTP status: 200
{
    "situs": "id.wikipedia.org",
    "url_robots": "https://id.wikipedia.org/robots.txt",
    "url_halaman": "https://id.wikipedia.org/wiki/Daftar_kabupaten_dan_kota_di_Nusa_Tenggara_Barat",
    "http_status_robots": 200,
    "waktu_pemeriksaan_utc": "2026-11-19T11:20:25.855269+00:00",
    "md5_robots_txt": "8dd94bf3d01cbb25e0af4e64c0485433",
    "user_agent": "UTS-DAM-NTB/1.0 (kontak: riska@mahasiswa.instiki.ac.id)",
    "can_fetch_user_agent": true,
    "can_fetch_wildcard": true,
    "crawl_delay": null,
    "jeda_yang_dipakai_detik": 1.5,
    "hasil": "DIIZINKAN"
}
URL: https://id.wikipedia.org/wiki/Daftar_kabupaten_dan_kota_di_Nusa_Tenggara_Barat | ID revisi: 29248605
Tabel 0: 10 baris x 10 kolom
Tabel 1: 4 baris x 3 kolom
Tabel 2: 7 baris x 3 kolom
Baris: 10 | Kolom: 12


,No.,Kabupaten/kota,Ibu kota,Bupati/wali kota,Luas wilayah (km2),Jumlah penduduk (2025),Kecamatan,Kelurahan/desa,Lambang alt,Peta lokasi,sumber_halaman,id_revisi_wikipedia
0,1,Kabupaten Bima,Woha,Ady Mahyudi,"4.208,861",541.582,18,-/191,NaN,NaN,https://id.wikipedia.org/wiki/Daftar_kabupaten...,29248605
1,2,Kabupaten Dompu,Dompu,Bambang Firdaus,"2.270,360",277.837,8,9/72,NaN,NaN,https://id.wikipedia.org/wiki/Daftar_kabupaten...,29248605
2,3,Kabupaten Lombok Barat,Gerung,Lalu Ahmad Zaini,919440,762.757,10,3/119,NaN,NaN,https://id.wikipedia.org/wiki/Daftar_kabupaten...,29248605
3,4,Kabupaten Lombok Tengah,Praya,Lalu Pathul Bahri,"1.169,029",1.128.716,12,12/127,NaN,NaN,https://id.wikipedia.org/wiki/Daftar_kabupaten...,29248605
4,5,Kabupaten Lombok Timur,Selong,Haerul Warisin,"1.607,377",1.457.591,20,15/239,NaN,NaN,https://id.wikipedia.org/wiki/Daftar_kabupaten...,29248605
5,6,Kabupaten Lombok Utara,Tanjung,Najmul Akhyar,810185,269.989,5,-/43,NaN,NaN,https://id.wikipedia.org/wiki/Daftar_kabupaten...,29248605
6,7,Kabupaten Sumbawa,Sumbawa,Syarafuddin Jarot,"6.645,723",527.715,24,8/157,NaN,NaN,https://id.wikipedia.org/wiki/Daftar_kabupaten...,29248605
7,8,Kabupaten Sumbawa Barat,Taliwang,Amar Nurmansyah,"1.739,564",156.059,8,7/57,NaN,NaN,https://id.wikipedia.org/wiki/Daftar_kabupaten...,29248605
8,9,Kota Bima,-,A. Rahman H. Abidin,208389,165.113,5,38/-,NaN,NaN,https://id.wikipedia.org/wiki/Daftar_kabupaten...,29248605
9,10,Kota Mataram,-,Mohan Roliskana,60063,461.936,6,50/-,NaN,NaN,https://id.wikipedia.org/wiki/Daftar_kabupaten...,29248605


Atribusi: Wikipedia Bahasa Indonesia, kontributor Wikipedia, https://id.wikipedia.org/wiki/Daftar_kabupaten_dan_kota_di_Nusa_Tenggara_Barat


In [30]:
# ===== SEL 7 (SOAL 5): MERGE FILE CSV + API CUACA =====
# Agregasi cuaca harian -> bulanan (kunci: Tahun, Bulan_Int)
cuaca_bulanan = (cuaca.groupby(["Tahun", "Bulan"], as_index=False)
    .agg(suhu_rata_rata_c=("suhu_rata_rata_c", "mean"),
         curah_hujan_total_mm=("curah_hujan_mm", "sum"),
         kecepatan_angin_maks_kmh=("kecepatan_angin_maks_kmh", "max"),
         jumlah_hari_data_cuaca=("tanggal", "count"))
    .rename(columns={"Bulan": "Bulan_Int"}))

# Kunci merge: gunakan data yang SUDAH DIBERSIHKAN dan samakan tipe numeriknya
kiri = df_bersih.copy()
kiri["Bulan_Int"] = kiri["Bulan"].map(bulan_map)
for kolom in ["Tahun", "Bulan_Int"]:
    kiri[kolom] = kiri[kolom].astype("int64")
    cuaca_bulanan[kolom] = cuaca_bulanan[kolom].astype("int64")

# Cek duplikasi SEBELUM merge pada kunci alami data utama
kunci = ["Tahun", "Bulan_Int", "Kategori Wisatawan"]
assert kiri.duplicated(subset=kunci).sum() == 0, "Ada duplikasi pada data utama."

# Merge: setiap baris kunjungan cocok dengan tepat 1 baris cuaca bulanan
hasil_merge = kiri.merge(cuaca_bulanan, on=["Tahun", "Bulan_Int"], how="left",
                         validate="many_to_one", indicator=True)

# Bukti tidak terjadi duplikasi dan tidak ada baris hilang/bertambah
print(hasil_merge["_merge"].value_counts())
dup_baris = int(hasil_merge.duplicated().sum())
dup_kunci = int(hasil_merge.duplicated(subset=kunci).sum())
print("Duplikasi seluruh baris:", dup_baris, "| Duplikasi kunci:", dup_kunci)
assert len(hasil_merge) == len(df_bersih)
assert (hasil_merge["_merge"] == "both").all()
assert dup_baris == 0 and dup_kunci == 0
assert hasil_merge["Jumlah (Orang)"].sum() == df_bersih["Jumlah (Orang)"].sum()

# Kelengkapan data cuaca: jumlah hari harus sama dengan jumlah hari kalender
hari_kalender = hasil_merge.apply(lambda r: calendar.monthrange(int(r["Tahun"]), int(r["Bulan_Int"]))[1], axis=1)
bulan_tidak_lengkap = int((hasil_merge["jumlah_hari_data_cuaca"] != hari_kalender).sum())
print("Baris dengan data cuaca tidak lengkap:", bulan_tidak_lengkap)

metadata_merge = {
    "sumber_1": URL_DATASET, "sumber_2": API_URL,
    "kunci_merge": ["Tahun", "Bulan (angka)"], "metode": "left merge", "validate": "many_to_one",
    "jumlah_baris_dataset_awal": int(len(df_bersih)), "jumlah_baris_hasil_merge": int(len(hasil_merge)),
    "jumlah_cocok": int((hasil_merge["_merge"] == "both").sum()),
    "jumlah_tidak_cocok": int((hasil_merge["_merge"] == "left_only").sum()),
    "duplikasi_baris": dup_baris, "duplikasi_kunci": dup_kunci,
    "baris_cuaca_tidak_lengkap": bulan_tidak_lengkap,
    "kolom_hasil": [c for c in hasil_merge.columns if c not in ("_merge", "Bulan_Int")],
    "catatan": "Cuaca dari 1 titik (Mataram), diagregasi bulanan; nilai cuaca sama untuk kedua kategori wisatawan.",
    "atribusi_cuaca": "Weather data by Open-Meteo.com (CC BY 4.0)",
    "tanggal_pembuatan_utc": datetime.now(timezone.utc).isoformat(),
}
# Buang kolom bantu, lalu SIMPAN hasil akhir + metadata
hasil_akhir = hasil_merge.drop(columns=["_merge", "Bulan_Int"])
hasil_akhir.to_csv("data/processed/kunjungan_wisatawan_ntb_dengan_cuaca.csv", index=False, encoding="utf-8")
Path("metadata/metadata_merge.json").write_text(json.dumps(metadata_merge, indent=4, ensure_ascii=False), encoding="utf-8")
display(hasil_akhir.head())

_merge
both          300
left_only       0
right_only      0
Name: count, dtype: int64
Duplikasi seluruh baris: 0 | Duplikasi kunci: 0
Baris dengan data cuaca tidak lengkap: 0


,#,Kode provinsi,Nama Provinsi,Kategori Wisatawan,Jumlah (Orang),Tahun,Bulan,suhu_rata_rata_c,curah_hujan_total_mm,kecepatan_angin_maks_kmh,jumlah_hari_data_cuaca
0,1,52,Nusa Tenggara Barat,Wisatawan Nusantara,132472,2026,Juni,26.213333,25.4,20.0,30
1,2,52,Nusa Tenggara Barat,Wisatawan Mancanegara,88315,2026,Juni,26.213333,25.4,20.0,30
2,3,52,Nusa Tenggara Barat,Wisatawan Nusantara,122674,2026,Mei,26.487097,72.3,20.2,31
3,4,52,Nusa Tenggara Barat,Wisatawan Mancanegara,81783,2026,Mei,26.487097,72.3,20.2,31
4,5,52,Nusa Tenggara Barat,Wisatawan Nusantara,113304,2026,April,26.513333,199.7,16.8,30


In [31]:
# ===== SEL 8: PROVENANCE (URL, TANGGAL AKSES ASLI, LISENSI, MD5) =====
source_info = {
    raw_dataset_path.name: {"url": URL_DATASET,
        "lisensi": "License Not Specified (Satu Data NTB); dipakai akademik nonkomersial, atribusi sumber"},
    raw_api_path.name: {"url": API_URL, "lisensi": "CC BY 4.0 (Open-Meteo) https://open-meteo.com/en/license"},
    "wikipedia_robots.txt": {"url": ROBOTS_URL, "lisensi": "Aturan crawler, bukan lisensi konten"},
    "wikipedia_ntb.html": {"url": SCRAPE_URL, "lisensi": "Tidak dinyatakan pada halaman; hanya tabel wilayah diambil untuk tugas akademik"},
}

# Hapus file sisa yang tidak terdaftar di source_info agar tidak mengacaukan pengujian
for p in Path("data/raw").glob("*"):
    if p.is_file() and p.name not in source_info:
        p.unlink()

# Pastikan SETIAP file di data/raw punya catatan provenance
file_raw = {p.name for p in Path("data/raw").glob("*") if p.is_file()}
assert file_raw == set(source_info), f"File raw tidak cocok dengan provenance: {file_raw ^ set(source_info)}"

provenance = pd.DataFrame([{
    "nama_file": n, "url_sumber": info["url"], "tanggal_akses_utc": log_akses.get(n, datetime.now(timezone.utc).isoformat()),
    "lisensi_keterangan": info["lisensi"],
    "ukuran_byte": (Path("data/raw") / n).stat().st_size,
    "md5": hitung_md5(Path("data/raw") / n),
} for n, info in source_info.items()])
display(provenance)
provenance.to_csv("metadata/provenance.csv", index=False)

,nama_file,url_sumber,tanggal_akses_utc,lisensi_keterangan,ukuran_byte,md5
0,data_20261119132258-jumlah-kunjungan-wisatawan...,https://data.ntbprov.go.id/dataset/9e270f88-a6...,2026-11-19T11:20:23.358413+00:00,License Not Specified (Satu Data NTB); dipakai...,24386,1a33870860c8f9abed2e168f96f7adee
1,open_meteo_ntb_historis.json,https://archive-api.open-meteo.com/v1/archive,2026-11-19T11:20:24.180295+00:00,CC BY 4.0 (Open-Meteo) https://open-meteo.com/...,128602,33d7ecb64e942bf5b3c4b9a1f89121d8
2,wikipedia_robots.txt,https://id.wikipedia.org/robots.txt,2026-11-19T11:20:25.855269+00:00,"Aturan crawler, bukan lisensi konten",15801,8dd94bf3d01cbb25e0af4e64c0485433
3,wikipedia_ntb.html,https://id.wikipedia.org/wiki/Daftar_kabupaten...,2026-11-19T11:20:27.391967+00:00,Tidak dinyatakan pada halaman; hanya tabel wil...,121083,99edf6bda31f63a8877e69739f2f873c


In [32]:
# ===== SEL 9: VERIFIKASI AKHIR + ZIP UNTUK DIUNGGAH =====
for folder in ["data/raw", "data/interim", "data/processed", "metadata"]:
    print(f"\n[{folder}]")
    for p in sorted(Path(folder).glob("*")):
        if p.is_file():
            print(f"- {p} ({p.stat().st_size} byte)")

assert len(df) >= 100 and len(cuaca) >= 30 and len(hasil_akhir) == len(df_bersih)
assert hasil_akhir.duplicated().sum() == 0
print("\nPemeriksaan dasar selesai.")

# Kemas hasil agar tidak hilang saat runtime Colab berakhir
with zipfile.ZipFile("hasil_uts_dam_ntb.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for folder in ["data", "metadata", "docs"]:
        for p in Path(folder).rglob("*"):
            if p.is_file():
                z.write(p)
from google.colab import files
files.download("hasil_uts_dam_ntb.zip")


[data/raw]
- data/raw/data_20261119132258-jumlah-kunjungan-wisatawan-berdasarkan-jenis-wisatawan.csv (24386 byte)
- data/raw/open_meteo_ntb_historis.json (128602 byte)
- data/raw/wikipedia_ntb.html (121083 byte)
- data/raw/wikipedia_robots.txt (15801 byte)

[data/interim]
- data/interim/cuaca_harian_ntb_historis.csv (147651 byte)
- data/interim/kunjungan_wisatawan_ntb_bersih.csv (19889 byte)
- data/interim/listing_pariwisata_ntb_scraping.csv (1153 byte)
- data/interim/tabel_wikipedia_ntb_scraping.csv (1768 byte)

[data/processed]
- data/processed/kunjungan_wisatawan_ntb_dengan_cuaca.csv (29763 byte)

[metadata]
- metadata/bukti_robots_indonesia_travel.json (621 byte)
- metadata/bukti_robots_wikipedia.json (577 byte)
- metadata/data_ethics_checklist.csv (2570 byte)
- metadata/data_ethics_checklist.xlsx (6355 byte)
- metadata/identifikasi_dataset.csv (591 byte)
- metadata/klasifikasi_kolom_pii.csv (558 byte)
- metadata/metadata_merge.json (1068 byte)
- metadata/provenance.csv (1014 byte

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>